<a href="https://colab.research.google.com/github/Kaykayag/acuvia/blob/main/acuvia_medgemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


In [ ]:
!pip install -q --upgrade transformers accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 21.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")  # ✅ safe

login(HF_TOKEN)

print("Login successful")

Login successful


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/medgemma-4b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (first run = ~8GB download)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Done! Device:", next(model.parameters()).device)

Loading tokenizer...


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Done! Device: cuda:0


In [ ]:
TRIAGE_SYSTEM_PROMPT = """You are a medical triage assistant in a mobile health app.
The patient will give you their demographics and symptoms.
Respond ONLY in this exact format — no extra text:

PRIORITY: [Non-Urgent / Urgent / Emergency]
TAGLINE: [max 5 words shown under the badge]
REASON: [1-2 sentences, mention age/sex if relevant]
NEXT_STEPS:
- [step 1]
- [step 2]
- [step 3]
DISCLAIMER: For guidance only - not a diagnosis

Priority rules:
- Emergency: life-threatening right now (chest pain + sweating, stroke, severe breathing difficulty)
- Urgent: serious but stable, needs care within hours
- Non-Urgent: mild, can monitor at home

IMPORTANT: Always respond in the same language the patient used."""

def run_triage(symptoms: list, free_text: str,
               age: int, sex: str, conditions: list) -> dict:

    symptom_str = ", ".join(symptoms) if symptoms else "none specified"
    condition_str = ", ".join(conditions) if conditions else "none"
    extra = f" Additional info: {free_text}" if free_text else ""

    user_msg = f"""Patient:
- Age: {age}, Sex: {sex}
- Known conditions: {condition_str}
- Symptoms: {symptom_str}.{extra}"""

    messages = [{"role": "user",
                 "content": TRIAGE_SYSTEM_PROMPT + "\n\n" + user_msg}]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
        )

    input_len = inputs["input_ids"].shape[1]
    raw = tokenizer.decode(outputs[0][input_len:],
                           skip_special_tokens=True).strip()

    result = {"priority": None, "tagline": None,
              "reason": None, "next_steps": [], "raw": raw}
    in_steps = False
    for line in raw.splitlines():
        line = line.strip()
        if line.startswith("PRIORITY:"):
            result["priority"] = line.replace("PRIORITY:", "").strip()
            in_steps = False
        elif line.startswith("TAGLINE:"):
            result["tagline"] = line.replace("TAGLINE:", "").strip()
            in_steps = False
        elif line.startswith("REASON:"):
            result["reason"] = line.replace("REASON:", "").strip()
            in_steps = False
        elif line.startswith("NEXT_STEPS:"):
            in_steps = True
        elif in_steps and line.startswith("-"):
            result["next_steps"].append(line.lstrip("- ").strip())

    return result

print("Triage function ready.")




Triage function ready.


In [ ]:
tests = [
    {
        "symptoms": ["Chest Pains", "Shortness of Breath"],
        "free_text": "also sweating heavily",
        "age": 58, "sex": "Male",
        "conditions": ["Hypertension"]
    },
    {
        "symptoms": ["Fever", "Nausea or Vomiting"],
        "free_text": "",
        "age": 25, "sex": "Female",
        "conditions": []
    },
    {
        "symptoms": ["Fever"],
        "free_text": "mild runny nose",
        "age": 30, "sex": "Male",
        "conditions": []
    },
    {
        "symptoms": ["itchy over my body"],
        "free_text": "my mody stinging when i scratch iy",
        "age": 21, "sex": "Female",
        "conditions": []

    },
    {
        "symptoms": ["shortness of breath","cough"],
        "free_text": "my plegm is green and hard",
        "age": 69, "sex": "Male",
        "conditions": []

    },
    {
        "symptoms": ["sakit kay akong ulo","labad akong ulo","kasuka on ko"],
        "free_text": "galisod ko ug hinga",
        "age": 8, "sex": "Male",
        "conditions": []
    }
]


for i, t in enumerate(tests):
    print(f"\n{'='*50}")
    print(f"TEST {i+1} | Age:{t['age']} {t['sex']} | {t['symptoms']}")
    print('='*50)
    r = run_triage(**t)
    print(f"PRIORITY : {r['priority']}")
    print(f"TAGLINE  : {r['tagline']}")
    print(f"REASON   : {r['reason']}")
    print(f"STEPS    : {r['next_steps']}")


TEST 1 | Age:58 Male | ['Chest Pains', 'Shortness of Breath']
PRIORITY : Urgent
TAGLINE  : Chest Pain, Shortness of Breath
REASON   : 58 year old male with hypertension experiencing chest pains and shortness of breath, accompanied by heavy sweating. This could indicate a cardiac issue requiring prompt evaluation.
STEPS    : ['Call emergency services immediately.', 'While waiting for EMS, have the patient sit or lie down comfortably.', 'Monitor vital signs (pulse, blood pressure, respiratory rate).']

TEST 2 | Age:25 Female | ['Fever', 'Nausea or Vomiting']
PRIORITY : Non-Urgent
TAGLINE  : Fever, Nausea, Vomiting
REASON   : 25 year old female with fever and nausea/vomiting.
STEPS    : ['Monitor temperature and hydration levels closely.', 'If vomiting persists or worsens, seek medical attention.', 'Consider over-the-counter medications for symptom relief.']

TEST 3 | Age:30 Male | ['Fever']
PRIORITY : Non-Urgent
TAGLINE  : Mild fever, runny nose
REASON   : 30-year-old male with mild fev

In [ ]:
# ── Cell 6: Install dependencies ──────────────────────────────────────
!pip install flask pyngrok flask-cors -q

from pyngrok import ngrok, conf
import os
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")  # ✅ safe
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print("ngrok configured.")

ngrok configured.


In [ ]:
# ── Cell 7: Start Flask API + ngrok tunnel ─────────────────────────────
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import torch
from pyngrok import ngrok
from google.colab import userdata

app = Flask(__name__)
CORS(app)  # allows your FastAPI backend to call this freely

# ── /health ────────────────────────────────────────────────────────────
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": "medgemma-4b-it"})

# ── /triage  ───────────────────────────────────────────────────────────
@app.route("/triage", methods=["POST"])
def triage():
    try:
        body = request.get_json(force=True)

        # Validate required fields
        required = ["symptoms", "age", "sex"]
        for field in required:
            if field not in body:
                return jsonify({"error": f"Missing field: {field}"}), 400

        result = run_triage(
            symptoms   = body.get("symptoms", []),
            free_text  = body.get("free_text", ""),
            age        = int(body["age"]),
            sex        = body["sex"],
            conditions = body.get("conditions", []),
        )

        return jsonify({
            "priority"   : result["priority"],
            "tagline"    : result["tagline"],
            "reason"     : result["reason"],
            "next_steps" : result["next_steps"],
        })

    # FIX: Added the missing exception handler for the triage endpoint!
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ── /chat ──────────────────────────────────────────────────────────────
@app.route("/chat", methods=["POST"])
def chat():
    try:
        body = request.get_json(force=True)
        messages = body.get("messages", [])

        if not messages:
            return jsonify({"error": "messages cannot be empty"}), 400

        # Build inputs using the chat template with full history
        inputs = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            return_dict=True,
            add_generation_prompt=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=True,
                temperature=0.7,       # slightly creative for conversation
                repetition_penalty=1.1,
            )

        input_len = inputs["input_ids"].shape[1]
        reply = tokenizer.decode(
            outputs[0][input_len:], skip_special_tokens=True
        ).strip()

        return jsonify({"reply": reply})

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ── Start server in background thread ─────────────────────────────────
def run_server():
    # use_reloader=False is crucial in Colab to prevent thread crashing
    app.run(port=5000, debug=False, use_reloader=False)

# Only start the thread if it isn't already running
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# ── Open ngrok tunnel ──────────────────────────────────────────────────
# Kill any existing tunnels first (safe to re-run)
ngrok.kill()

# Apply the token safely using our fix from earlier
try:
    ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
except Exception as e:
    print("Warning: Could not load NGROK_AUTHTOKEN. Make sure it is set in the Secrets tab.")

public_url = ngrok.connect(5000).public_url

print("=" * 55)
print(f"  ✅ MedGemma API is live!")
print(f"  🌐 URL   : {public_url}")
print(f"  POST   : {public_url}/triage")
print(f"  POST   : {public_url}/chat")
print(f"  GET    : {public_url}/health")
print("=" * 55)
print("\n📋 Copy the URL above → paste into your backend .env")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


  ✅ MedGemma API is live!
  🌐 URL   : https://moneybags-barterer-catcall.ngrok-free.dev
  POST   : https://moneybags-barterer-catcall.ngrok-free.dev/triage
  POST   : https://moneybags-barterer-catcall.ngrok-free.dev/chat
  GET    : https://moneybags-barterer-catcall.ngrok-free.dev/health

📋 Copy the URL above → paste into your backend .env
